In [1]:
import os

save_path = os.path.join(os.getcwd(), "app.py")

app_code = """import os
import re
import math
import difflib
from collections import defaultdict, Counter
from datetime import datetime
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from flask import Flask, request, jsonify, render_template_string

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)

STOP_WORDS = set(stopwords.words("english"))
STEMMER    = PorterStemmer()
LEMMATIZER = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\\s]", " ", text)
    text = re.sub(r"\\s+", " ", text).strip()
    return text

def tokenize(text):
    return re.findall(r"\\b[a-z0-9]+\\b", text)

def preprocess(text):
    tokens = tokenize(clean_text(text))
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [STEMMER.stem(t) for t in tokens]
    return tokens

def load_documents(data_dir):
    docs = {}
    metadata = {}
    doc_id = 0
    if not os.path.exists(data_dir):
        print(f"ERROR: {data_dir} not found!")
        return docs, metadata
    for filename in sorted(os.listdir(data_dir)):
        if not filename.endswith(".txt"):
            continue
        path = os.path.join(data_dir, filename)
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read().strip()
        if not text:
            continue
        name_parts = os.path.splitext(filename)[0].replace("_", " ").split()
        category = name_parts[0].capitalize() if name_parts else "General"
        docs[doc_id] = text
        metadata[doc_id] = {
            "title"     : os.path.splitext(filename)[0].replace("_", " ").title(),
            "filename"  : filename,
            "category"  : category,
            "word_count": len(text.split()),
        }
        doc_id += 1
    return docs, metadata

def build_index(docs):
    index = defaultdict(dict)
    doc_lengths = {}
    for doc_id, text in docs.items():
        tokens = preprocess(text)
        counts = Counter(tokens)
        doc_lengths[doc_id] = sum(counts.values())
        for term, tf in counts.items():
            index[term][doc_id] = tf
    return dict(index), doc_lengths

class BM25Ranker:
    def __init__(self, index, doc_lengths, k1=1.5, b=0.75):
        self.index = index
        self.doc_lengths = doc_lengths
        self.N = len(doc_lengths)
        self.avgdl = sum(doc_lengths.values()) / float(self.N) if self.N > 0 else 1.0
        self.k1 = k1
        self.b  = b
        self.idf = self._compute_idf()

    def _compute_idf(self):
        idf = {}
        for term, postings in self.index.items():
            df = len(postings)
            idf[term] = math.log(1 + (self.N - df + 0.5) / (df + 0.5))
        return idf

    def score(self, query_terms):
        scores = {}
        for term in query_terms:
            if term not in self.index:
                continue
            idf = self.idf.get(term, 0.0)
            for doc_id, tf in self.index[term].items():
                dl    = self.doc_lengths[doc_id]
                denom = tf + self.k1 * (1.0 - self.b + self.b * dl / self.avgdl)
                s     = idf * tf * (self.k1 + 1.0) / denom
                scores[doc_id] = scores.get(doc_id, 0.0) + s
        return scores

def get_postings(term):
    processed = preprocess(term)
    if not processed:
        return set()
    return set(index.get(processed[0], {}).keys())

def boolean_search(query):
    parts = re.split(r"\\b(AND|OR|NOT)\\b", query.upper())
    tokens = [p.strip() for p in parts if p.strip()]
    if not tokens:
        return ALL_DOCS.copy()
    result = None
    current_op = "OR"
    for token in tokens:
        if token in ("AND", "OR", "NOT"):
            current_op = token
            continue
        docs_for_term = get_postings(token)
        if result is None:
            result = docs_for_term
        elif current_op == "AND":
            result = result & docs_for_term
        elif current_op == "OR":
            result = result | docs_for_term
        elif current_op == "NOT":
            result = result - docs_for_term
    return result if result is not None else set()

def autocomplete(prefix, max_suggestions=5):
    prefix = prefix.lower().strip()
    if not prefix:
        return []
    return [w for w in all_raw_words if w.startswith(prefix)][:max_suggestions]

def spell_correct(word, cutoff=0.6):
    word = word.lower().strip()
    stem = STEMMER.stem(word)
    if stem in set(index.keys()):
        return word
    raw_matches = difflib.get_close_matches(word, all_raw_words, n=3, cutoff=cutoff)
    if raw_matches:
        return raw_matches[0]
    return word

def correct_query(query):
    boolean_ops = {"and", "or", "not"}
    words = query.lower().split()
    corrected = []
    corrections = {}
    for word in words:
        if word in boolean_ops:
            corrected.append(word.upper())
            continue
        suggestion = spell_correct(word)
        if suggestion != word:
            corrections[word] = suggestion
        corrected.append(suggestion)
    return " ".join(corrected), len(corrections) > 0, corrections

def generate_snippet(doc_text, query, max_len=250):
    raw_terms = [
        w for w in re.findall(r"\\b[a-z0-9]+\\b", query.lower())
        if w not in {"and", "or", "not"} and w not in STOP_WORDS
    ]
    if not raw_terms:
        return doc_text[:max_len] + ("..." if len(doc_text) > max_len else "")
    sentences  = re.split(r"(?<=[.!?])\\s+", doc_text)
    best_sent  = doc_text[:max_len]
    best_count = 0
    for sentence in sentences:
        count = sum(1 for t in raw_terms if t in sentence.lower())
        if count > best_count:
            best_count = count
            best_sent  = sentence
    if len(best_sent) > max_len:
        best_sent = best_sent[:max_len]
    highlighted = best_sent
    for term in raw_terms:
        pattern = re.compile(re.escape(term), re.IGNORECASE)
        highlighted = pattern.sub(lambda m: f"<mark>{m.group(0)}</mark>", highlighted)
    return highlighted.strip() + ("..." if len(doc_text) > max_len else "")

def search_engine(query, top_k=5, category=None):
    if not query.strip():
        return []
    corrected_q, _, _ = correct_query(query)
    upper_words = corrected_q.upper().split()
    has_bool    = any(op in upper_words for op in ["AND", "OR", "NOT"])
    clean_q     = re.sub(r"\\b(AND|OR|NOT)\\b", "", corrected_q, flags=re.IGNORECASE)
    tokens      = preprocess(clean_q)
    scores      = ranker.score(tokens)
    if has_bool:
        candidate_docs = boolean_search(corrected_q)
        scores = {d: s for d, s in scores.items() if d in candidate_docs}
        for d in candidate_docs:
            if d not in scores:
                scores[d] = 0.01
    if category:
        scores = {
            d: s for d, s in scores.items()
            if doc_metadata.get(d, {}).get("category", "").lower() == category.lower()
        }
    ranked  = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_k]
    results = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        meta = doc_metadata[doc_id]
        results.append({
            "rank"      : rank,
            "doc_id"    : doc_id,
            "title"     : meta["title"],
            "category"  : meta["category"],
            "word_count": meta["word_count"],
            "score"     : round(score, 4),
            "snippet"   : generate_snippet(docs[doc_id], corrected_q),
        })
    return results

HTML_TEMPLATE = \"\"\"
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Intelligent Search Engine</title>
  <style>
    * { box-sizing: border-box; margin: 0; padding: 0; }
    body { font-family: Arial, sans-serif; background: #f8f9fa; color: #202124; }
    .header { background: white; padding: 15px 30px; border-bottom: 1px solid #dfe1e5;
              display: flex; align-items: center; gap: 20px; position: sticky; top: 0;
              z-index: 100; box-shadow: 0 2px 4px rgba(0,0,0,0.1); }
    .logo { font-size: 24px; font-weight: bold; color: #1a73e8; text-decoration: none; }
    .search-form { display: flex; flex: 1; max-width: 700px; }
    .search-input { flex: 1; padding: 12px 20px; font-size: 16px;
                    border: 2px solid #dfe1e5; border-radius: 24px 0 0 24px; outline: none; }
    .search-input:focus { border-color: #1a73e8; }
    .search-btn { padding: 12px 24px; background: #1a73e8; color: white; border: none;
                  border-radius: 0 24px 24px 0; cursor: pointer; font-size: 16px; font-weight: bold; }
    .search-btn:hover { background: #1557b0; }
    .home-container { display: flex; flex-direction: column; align-items: center;
                      justify-content: center; min-height: 80vh; gap: 30px; }
    .home-logo { font-size: 60px; font-weight: bold; color: #1a73e8; }
    .home-search-form { display: flex; width: 600px; max-width: 90vw; }
    .home-search-input { flex: 1; padding: 15px 25px; font-size: 18px;
                         border: 2px solid #dfe1e5; border-radius: 30px 0 0 30px; outline: none; }
    .home-search-input:focus { border-color: #1a73e8; }
    .home-search-btn { padding: 15px 30px; background: #1a73e8; color: white; border: none;
                       border-radius: 0 30px 30px 0; cursor: pointer; font-size: 18px; font-weight: bold; }
    .tips { color: #666; font-size: 14px; text-align: center; line-height: 1.8; }
    .results-container { max-width: 750px; margin: 30px auto; padding: 0 20px; }
    .result-count { color: #70757a; font-size: 14px; margin-bottom: 20px; }
    .corrected-notice { background: #fff3cd; border: 1px solid #ffc107; border-radius: 8px;
                        padding: 10px 15px; margin-bottom: 15px; font-size: 14px; color: #856404; }
    .result-card { background: white; border-radius: 12px; padding: 20px; margin-bottom: 15px;
                   box-shadow: 0 1px 4px rgba(0,0,0,0.1); transition: box-shadow 0.2s; }
    .result-card:hover { box-shadow: 0 4px 12px rgba(0,0,0,0.15); }
    .result-rank { display: inline-block; background: #1a73e8; color: white; border-radius: 50%;
                   width: 28px; height: 28px; text-align: center; line-height: 28px;
                   font-size: 13px; font-weight: bold; margin-right: 10px; }
    .result-title { font-size: 20px; color: #1a0dab; font-weight: bold; display: inline; }
    .result-meta { color: #70757a; font-size: 13px; margin: 8px 0;
                   display: flex; gap: 15px; flex-wrap: wrap; }
    .result-meta span { background: #f1f3f4; padding: 2px 8px; border-radius: 12px; }
    .result-snippet { color: #3c4043; font-size: 15px; line-height: 1.6; margin-top: 8px; }
    mark { background: #fff176; padding: 0 2px; border-radius: 2px; font-weight: bold; }
    .no-results { text-align: center; padding: 60px 20px; color: #666; }
    .filters { display: flex; gap: 10px; margin-bottom: 20px; flex-wrap: wrap; }
    .filter-btn { padding: 6px 16px; border: 1px solid #dfe1e5; border-radius: 20px;
                  background: white; cursor: pointer; font-size: 14px; color: #3c4043;
                  text-decoration: none; transition: all 0.2s; }
    .filter-btn:hover, .filter-btn.active { background: #1a73e8; color: white; border-color: #1a73e8; }
    .footer { text-align: center; padding: 30px; color: #70757a; font-size: 13px;
              border-top: 1px solid #dfe1e5; margin-top: 40px; }
  </style>
</head>
<body>
{% if query %}
<div class="header">
  <a class="logo" href="/">SearchEngine</a>
  <form class="search-form" method="GET" action="/">
    <input class="search-input" type="text" name="q"
           value="{{ query|e }}" placeholder="Search...">
    <button class="search-btn" type="submit">Search</button>
  </form>
</div>
<div class="results-container">
  {% if corrected and corrected != query %}
  <div class="corrected-notice">
    Showing results for <strong>{{ corrected }}</strong>
    &mdash; Original: <em>{{ query }}</em>
  </div>
  {% endif %}
  <div class="filters">
    <a class="filter-btn {% if not category %}active{% endif %}"
       href="/?q={{ query|e }}">All</a>
    {% for cat in categories %}
    <a class="filter-btn {% if category == cat %}active{% endif %}"
       href="/?q={{ query|e }}&category={{ cat }}">{{ cat }}</a>
    {% endfor %}
  </div>
  <div class="result-count">
    {{ results|length }} result(s) for
    <strong>"{{ corrected or query }}"</strong>
  </div>
  {% if results %}
    {% for r in results %}
    <div class="result-card">
      <div>
        <span class="result-rank">{{ r.rank }}</span>
        <span class="result-title">{{ r.title }}</span>
      </div>
      <div class="result-meta">
        <span>Category: {{ r.category }}</span>
        <span>Words: {{ r.word_count }}</span>
        <span>Score: {{ r.score }}</span>
      </div>
      <div class="result-snippet">{{ r.snippet|safe }}</div>
    </div>
    {% endfor %}
  {% else %}
  <div class="no-results">
    <h2>No results found for "{{ query }}"</h2>
    <p>Try different keywords.</p>
  </div>
  {% endif %}
</div>
{% else %}
<div class="home-container">
  <div class="home-logo">SearchEngine</div>
  <form class="home-search-form" method="GET" action="/">
    <input class="home-search-input" type="text" name="q"
           placeholder="Search documents..." autofocus>
    <button class="home-search-btn" type="submit">Search</button>
  </form>
  <div class="tips">
    Use <code>AND</code> | <code>OR</code> | <code>NOT</code> for Boolean search<br>
    Example: <em>machine AND learning</em> | <em>database NOT sql</em>
  </div>
</div>
{% endif %}
<div class="footer">
  Intelligent Document Search Engine &mdash;
  Built with Python, Flask &amp; BM25
</div>
</body>
</html>
\"\"\"

app = Flask(__name__)

@app.route("/")
def home():
    query      = request.args.get("q", "").strip()
    category   = request.args.get("category", "").strip()
    results    = []
    corrected  = query
    categories = sorted(set(m["category"] for m in doc_metadata.values()))
    if query:
        corrected, _, _ = correct_query(query)
        results = search_engine(
            corrected, top_k=10,
            category=category if category else None
        )
    return render_template_string(
        HTML_TEMPLATE,
        query      = query,
        corrected  = corrected,
        results    = results,
        categories = categories,
        category   = category
    )

@app.route("/api/search")
def api_search():
    q       = request.args.get("q", "").strip()
    top_k   = int(request.args.get("k", 5))
    results = search_engine(q, top_k=top_k)
    return jsonify({"query": q, "count": len(results), "results": results})

@app.route("/api/suggest")
def api_suggest():
    prefix = request.args.get("q", "")
    return jsonify({"suggestions": autocomplete(prefix)})

@app.route("/api/stats")
def api_stats():
    return jsonify({
        "total_documents": len(docs),
        "vocabulary_size": len(index),
        "categories": list(set(m["category"] for m in doc_metadata.values()))
    })

if __name__ == "__main__":
    data_dir = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "docs"
    )
    print("Loading documents...")
    docs, doc_metadata = load_documents(data_dir)
    print(f"  Loaded {len(docs)} documents")
    print("Building index...")
    index, doc_lengths = build_index(docs)
    ALL_DOCS = set(docs.keys())
    print(f"  Vocabulary: {len(index):,} terms")
    print("Creating BM25 ranker...")
    ranker = BM25Ranker(index, doc_lengths)
    print("Building word list...")
    all_raw_words = sorted(set(
        w for doc_text in docs.values()
        for w in re.findall(r"\\b[a-z]+\\b", doc_text.lower())
        if w not in STOP_WORDS and len(w) > 2
    ))
    print("\\n=============================================")
    print("  Search Engine is running!")
    print("  Open: http://127.0.0.1:5000")
    print("=============================================\\n")
    app.run(debug=True, port=5000)
"""

# Write the file
with open(save_path, "w", encoding="utf-8") as f:
    f.write(app_code)

print(f"app.py saved to : {save_path}")
print(f"File size       : {os.path.getsize(save_path):,} bytes")
print()
print("Now open Terminal and type these TWO commands:")
print()
print(f"cd {os.getcwd()}")
print("python app.py")

app.py saved to : /Users/aditrisingh/app.py
File size       : 16,085 bytes

Now open Terminal and type these TWO commands:

cd /Users/aditrisingh
python app.py
